# Autoencoder 02: Inference and latent-space images

A trained autoencoder exposes `encode`, `decode`, and whole-image `transform`. Latent values are saved as a new imzML/ibd pair with the original spatial coordinates and metadata, but with the spectral axis replaced by latent-component indices. This makes latent data readable and sliceable through the same image-like API.

In [1]:
import os
from pathlib import Path

repository_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
)
os.chdir(repository_root)
repository_root

'/home/maxi7524/repositories/MSIAutoEncoderWrapper'

In [3]:
from pathlib import Path
import numpy as np
import torch
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper
from msi_autoencoder_wrapper.models.datasets.strategies.pixel_dataset import PixelDataset

wrapper = MSIAutoEncoderWrapper("data/tutorial_workspace")
image_path = Path("data/tutorial_workspace/datasets/example_1/example_1.imzML").resolve()
wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
wrapper.context_manager.set_binner("LinearBinning", str(image_path), bin_step=0.1)
wrapper.context_manager.set_inverse_binner(
    "TopPeaksInverseBinner", str(image_path), max_bins=1500, window_size=3
)
wrapper.workspace.set_active_image(str(image_path))

2026-07-19 19:02:08,226 | INFO     | msi_autoencoder_wrapper.core.wrapper:70 | MSIAutoEncoderWrapper: Anchoring processing device state: cuda
2026-07-19 19:02:08,228 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:46 | Enforcing automatic module discovery for reader and binner registries.
2026-07-19 19:02:08,229 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 2 implementation module(s) in package 'msi_autoencoder_wrapper.readers.strategies'.
2026-07-19 19:02:08,230 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.binners_strategies'.
2026-07-19 19:02:08,231 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.inverse_strategies'.
2026-07-19 19:02:08,233 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 10 implementati

## Loaded model versus image-local model

`models_manager.model_functionality` belongs to the one currently loaded model. `active_context.local_model_functionality` belongs to the selected image. During `attach_model` or `load_model`, the manager first creates an `AutoencoderContextInterface` around the PyTorch model. With `bind_to_local_context=True`, that same interface is written to the active image bucket in `context_manager.config_ledger` and cached by `active_context`. Loading another manager model replaces only the manager reference; the image bucket still points to its previous autoencoder. Binding is optional because it deliberately keeps that model alive in memory.

In [4]:
# Run after the example model bundle is installed in the workspace.
wrapper.models_manager.load_model(
    img_name="example",
    model_name="example-autoencoder",
    bind_to_local_context=True,
)
loaded_autoencoder = wrapper.models_manager.model_functionality
local_autoencoder = wrapper.active_context.local_model_functionality
assert loaded_autoencoder is local_autoencoder

2026-07-19 19:02:29,756 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 20 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures'.
2026-07-19 19:02:29,756 | INFO     | msi_autoencoder_wrapper.models.model_loader:65 | Reconstructing loaded model family 'autoencoder'.
2026-07-19 19:02:29,758 | INFO     | msi_autoencoder_wrapper.models.architectures.architectures_manager:125 | Initializing multi-component sub-graph resolution phase for model family: autoencoder
2026-07-19 19:02:29,758 | INFO     | msi_autoencoder_wrapper.models.architectures.architectures_manager:144 | Instantiating standard component sub-module: Category='encoder' using Strategy='CNNEncoder'.
2026-07-19 19:02:29,764 | INFO     | msi_autoencoder_wrapper.models.architectures.architectures_manager:144 | Instantiating standard component sub-module: Category='decoder' using Strategy='CNNDecoder'.
2026-07-19 19:02:29,767 | INFO     | msi_autoencoder_wrapper.models.architectu

Loading a replacement with `bind_to_local_context=False` changes only the manager interface. `wrapper.active_context.autoencoder` prefers the local binding, while `wrapper.models_manager.autoencoder` always addresses the currently loaded model. This distinction is important when comparing models without losing an image's established transform.

In [ ]:
wrapper.models_manager.load_model(
    img_name="example",
    model_name="comparison-autoencoder",
    bind_to_local_context=False,
)
assert wrapper.active_context.autoencoder is local_autoencoder
assert wrapper.models_manager.autoencoder is not local_autoencoder

## Encode and decode explicit batches

`encode` expects binned `[batch, bins]` data. `decode(..., grid_xs=True)` returns the regular binned grid. The default `grid_xs=False` applies the configured inverse binner and returns sparse `(m/z, intensity)` pairs. Both methods switch to evaluation mode and disable gradient tracking.

In [5]:
xs, ys = wrapper.active_context.reader[0]
grid = wrapper.active_context.binner(xs=xs, ys=ys)
# autoencoder = wrapper.active_context.autoencoder
# latent_vector = autoencoder.encode(grid[None, :])
# reconstructed_grid = autoencoder.decode(latent_vector, grid_xs=True)
# reconstructed_sparse = autoencoder.decode(latent_vector, grid_xs=False)
# print(latent_vector.shape, reconstructed_grid.shape)

2026-07-19 19:02:34,175 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:108 | Successfully bound active context memory maps for: example


## Transform and save a complete latent image

Whole-image transform needs an active dataset associated with the current image. A loaded model can be used without its original image, but transformation cannot start until an image dataset is supplied. `save_latent` writes both `.imzML` and `.ibd`, then can activate the new latent reader without unloading the original reader.

In [6]:
wrapper.workspace.set_active_image(str(image_path))
wrapper.active_dataset = PixelDataset(
    active_context=wrapper.active_context,
    source="image",
)
latent_matrix = wrapper.active_context.autoencoder.transform(
    {"batch_size": 128, "num_workers": 0, "pin_memory": "device" == "cuda"}
)
latent_path = wrapper.active_context.save_latent(
    output_path="data/tutorial_workspace/models/example/example-autoencoder/latent/example.latent.imzML",
    loader_config={"batch_size": 128, "num_workers": 0},
    activate=True,
)

2026-07-19 19:02:44,700 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:75 | Active image context mapped by index key: example
2026-07-19 19:02:44,702 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.autoencoder_context_manager:165 | Initiating sequential image feature mapping over active data stream channels.
2026-07-19 19:03:05,493 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.autoencoder_context_manager:170 | Sequential structural image data translation complete.
2026-07-19 19:03:05,507 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.autoencoder_context_manager:165 | Initiating sequential image feature mapping over active data stream channels.
2026-07-19 19:03:25,994 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.autoencoder_context_manager:170 | Sequential structural image data translation complete.
2026-07-19 19:03:29,324 | INFO     | msi_autoencoder_wrapper.latent.imzml_

## Work on image and latent spaces independently

`data_source` controls default image-like operations, while an explicit `source=` avoids hidden switching. The original and latent readers are not both loaded on demand: each is resolved only when requested, and the original may be absent in a latent-only workflow. Coordinate order and slice semantics are identical in both spaces.

In [8]:
wrapper.active_context.set_data_source("latent")
latent_spectrum = wrapper.active_context.get_spectrum(0, source="latent")
latent_region = wrapper.active_context.get_region(
    slice(10, 20), slice(30, 40), source="latent"
)
original_region = wrapper.active_context.get_region(
    slice(10, 20), slice(30, 40), source="image"
)
latent_dataset = PixelDataset(active_context=wrapper.active_context, source="latent")

A latent-only consumer can create an empty workspace and call `active_context.load_latent(path)`. It can slice latent data and build `PixelDataset(source="latent")` without opening the original image. Decoding with a loaded autoencoder also works with `grid_xs=True`; sparse decoding additionally needs the matching inverse binner.

## Monitor memory

Run the commands available on the current platform. Linux commands also work in WSL. `nvidia-smi` is supplied with the NVIDIA driver; it is not useful for Apple MPS. macOS provides `vm_stat` and `memory_pressure`. `psutil` is optional (`pip install psutil`) because the library has a standard-library RAM fallback.

In [ ]:
# Linux / WSL:
# !free -h
# !df -h .
# NVIDIA GPU:
# !nvidia-smi
# macOS:
# !vm_stat
# !memory_pressure
# !df -h .

In [7]:
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print({"free_vram": free_bytes, "total_vram": total_bytes})
    print(torch.cuda.memory_summary(abbreviated=True))

{'free_vram': 5277483008, 'total_vram': 6442188800}
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   8931 KiB | 136396 KiB | 115686 MiB | 115678 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   8931 KiB | 136396 KiB | 115686 MiB | 115678 MiB |
|---------------------------------------------------------------------------|
| Requested memory      |   8923 KiB | 136388 KiB | 115686 MiB | 115677 MiB |
|-----------

For predictable memory use, begin with the estimator from Autoencoder 01, use a smaller transform batch, keep `num_workers=0` on constrained machines, avoid binding models locally unless persistence is needed, and process spatial slices rather than materializing every spectrum. `unload_latent()` releases the latent reader; `models_manager.unload_model()` releases only the loaded-model reference. A local binding intentionally keeps its model alive. `torch.cuda.empty_cache()` releases unused cached CUDA blocks, not tensors that still have references.

In [ ]:
# wrapper.active_context.unload_latent()
# wrapper.models_manager.unload_model()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()